# Lab 04: Functional Dependency Closure & BCNF Checker

**Z2004 Database Management Systems (BS Core) · IIT Madras Zanzibar**

Armstrong attribute closure, minimal candidate keys, BCNF violation detector, and lossless join check.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z2004/labs/DB04_Normalization_FD_and_BCNF_Checker.ipynb)

Run cells top to bottom (`Runtime > Run all`). Built on Python `sqlite3` with full self-checks.

## 1. Functional Dependency Closure, Candidate Keys & BCNF Verification

A relation $R$ is in Boyce-Codd Normal Form (BCNF) iff for every non-trivial FD $X \to Y$ holding on $R$, $X$ is a superkey ($R \subseteq X^+$).

In [1]:
from itertools import combinations


def closure(attrs, fds):
    cl = set(attrs)
    changed = True
    while changed:
        changed = False
        for lhs, rhs in fds:
            if lhs.issubset(cl) and not rhs.issubset(cl):
                cl.update(rhs); changed = True
    return frozenset(cl)


def candidate_keys(rel, fds):
    keys = []
    for k in range(1, len(rel) + 1):
        for comb in combinations(sorted(rel), k):
            c = frozenset(comb)
            if not any(old.issubset(c) for old in keys) and closure(c, fds) == rel:
                keys.append(c)
    return keys


def bcnf_violations(rel, fds):
    return [(l, r) for l, r in fds if l.issubset(rel) and r.issubset(rel) and not r.issubset(l) and not rel.issubset(closure(l, fds))]


R = frozenset({"A", "B", "C", "D"})
F = [(frozenset({"A"}), frozenset({"B"})), (frozenset({"B"}), frozenset({"C"}))]
print("Candidate keys:", [sorted(k) for k in candidate_keys(R, F)])
print("BCNF violations on R:", [(sorted(l), sorted(r)) for l, r in bcnf_violations(R, F)])
assert candidate_keys(R, F) == [frozenset({"A", "D"})]


Candidate keys: [['A', 'D']]
BCNF violations on R: [(['A'], ['B']), (['B'], ['C'])]
